# Gamma Scalping and Volatility Analysis

## 1. Project Objective

This notebook implements and evaluates a delta-hedged SPX straddle strategy using E-mini S&P 500 (ES) futures as the hedge instrument.

The analysis focuses on how gamma gains, theta decay, implied versus realised (subsequently) volatility, hedge frequency and transaction costs interact to determine strategy P&L (profit and loss).

In [ ]:
# ES Gamma Scalping (Daily) — March 10–20, 2020

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statistics import NormalDist

In [ ]:
plt.rcParams["figure.figsize"] = (10, 4) # set as default plot size

## 2. Model Configuration

This strategy models a long SPX straddle dynamically delta-hedged using E-mini S&P 500 futures. SPX determines the option exposure, while ES provides the tradable hedge instrument. The configuration below defines the contract multipliers, strategy assumptions, expiry, data period and volatility proxy used throughout the backtest.

In [ ]:
# Contract specifications
SPX_OPTION_MULTIPLIER = 100.0
ES_POINT_VALUE        = 50.0

# Strategy specification
N_STRADDLES            = 1
DELTA_REHEDGE_BAND     = 0.10
COMMISSION_PER_ES      = 1.25

# Risk controls
DAILY_PROFIT_TARGET    = 2000.0
DAILY_DRAWDOWN_LIMIT   = -2000.0

# Option assumptions
RISK_FREE_RATE         = 0.0
OPTION_EXPIRY          = pd.Timestamp("2020-03-20")
STRIKE_INTERVAL        = 5.0

# Data
SPX_TICKER             = "^GSPC"
ES_TICKER              = "ES=F"
VOL_TICKER             = "^VIX9D"

INTERVAL                = "1d"
START                   = "2020-03-10"
END                     = "2020-03-21"  # yfinance end date is exclusive

## 3. Data collection and preparation

In [ ]:
# =========================
# HELPERS
# =========================
def infer_es_contract_from_date(dt: pd.Timestamp) -> str:
    qmap = {3: ("H", 3), 6: ("M", 6), 9: ("U", 9), 12: ("Z", 12)} #4Quarters futures can expire in
    month = dt.month #extract month number from date
    year2 = dt.year % 100 #take last 2 digits of yr
    quarters = [3,6,9,12] #define 4 quarter expiry points
    target_q = None
    for q in quarters: #this loop iterates through 3,6,9,12
        if month <= q:
            target_q = q
            break
    if target_q is None: #if date after Dec (>12), loop around
        target_q = 3
        year2 = (dt.year + 1) % 100
    code = qmap[target_q][0]
    return f"ES{code}{year2:02d}.CME"

In [ ]:
def flatten_first_numeric_col(df, target_name): #flatten multiindex into single column names.
    if df is None or df.empty:
        return pd.DataFrame(columns=[target_name])
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join([str(c) for c in col if c]) for col in df.columns]
    first_col = df.columns[0]
    df = df[[first_col]].rename(columns={first_col: target_name})
    df.index = pd.to_datetime(df.index).tz_localize(None)
    return df

In [ ]:
def download_es_daily_with_fallback(start, end, contract_override=None): #download daily ES futures function.
    if contract_override:
        tickers = [contract_override]
    else:
        start_dt = pd.to_datetime(start)
        tickers = ["ES=F", infer_es_contract_from_date(start_dt)]
    for tk in tickers:
        es = yf.download(tk, interval="1d", start=start, end=end, progress=False)
        if es is not None and not es.empty:
            return es, tk
    return pd.DataFrame(), tickers[-1]

In [ ]:
def reindex_vix_to_es(es_df, vix_df):
    vix_d = vix_df.resample("1D").last()
    es_dates = pd.to_datetime(es_df.index.date)
    vix_on_es = vix_d.reindex(es_dates).ffill()
    vix_on_es.index = es_df.index
    return vix_on_es

In [ ]:
# =========================
# DOWNLOAD & PREP
# =========================
es_raw, used_ticker = download_es_daily_with_fallback(START, END, CONTRACT_OVERRIDE)
if es_raw.empty:
    raise ValueError("No ES data returned.")

vix_raw = yf.download("^VIX", interval="1d", start=START, end=END, progress=False)

es = flatten_first_numeric_col(es_raw, "ES")
vix = flatten_first_numeric_col(vix_raw, "VIX")

es["day"] = es.index.date
if not vix.empty:
    vix_on_es = reindex_vix_to_es(es, vix)
    es["VIX"] = vix_on_es["VIX"].values
else:
    es["VIX"] = 80.0

es["sigma"] = es["VIX"] / 100.0
es["K"]     = es.groupby("day")["ES"].transform("first")  # ATM strike = open
es["T"]     = T_CONST_DAYS / 365.0

bars_per_day = 1
dt_year = 1/252

print(f"Used ES ticker: {used_ticker}")
display(es.head())

## 4. Option pricing and Greeks

In [ ]:
# BLACK–SCHOLES GREEKS
# =========================
N = NormalDist(0, 1)

def d1(S, K, r, sigma, T):
    return (np.log(S/K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))

def d2(d1_, sigma, T):
    return d1_ - sigma * np.sqrt(T)

def gamma_bs(S, K, r, sigma, T):
    d1_ = d1(S, K, r, sigma, T)
    pdf = np.exp(-0.5 * d1_**2) / np.sqrt(2*np.pi)
    return pdf / (S * sigma * np.sqrt(T))

def delta_call(S, K, r, sigma, T): return N.cdf(d1(S, K, r, sigma, T))
def delta_put(S, K, r, sigma, T):  return N.cdf(d1(S, K, r, sigma, T)) - 1

def theta_call(S, K, r, sigma, T):
    d1_ = d1(S, K, r, sigma, T); d2_ = d2(d1_, sigma, T)
    pdf = np.exp(-0.5 * d1_**2) / np.sqrt(2*np.pi)
    return -(S * pdf * sigma) / (2 * np.sqrt(T)) - r*K*np.exp(-r*T)*N.cdf(d2_)

def theta_put(S, K, r, sigma, T):
    d1_ = d1(S, K, r, sigma, T); d2_ = d2(d1_, sigma, T)
    pdf = np.exp(-0.5 * d1_**2) / np.sqrt(2*np.pi)
    return -(S * pdf * sigma) / (2 * np.sqrt(T)) + r*K*np.exp(-r*T)*N.cdf(-d2_)

def straddle_greeks(S, K, sigma, T, n=1.0):
    d_c = delta_call(S, K, RISK_FREE, sigma, T)
    d_p = delta_put (S, K, RISK_FREE, sigma, T)
    g   = gamma_bs  (S, K, RISK_FREE, sigma, T)
    t   = theta_call(S, K, RISK_FREE, sigma, T) + theta_put(S, K, RISK_FREE, sigma, T)
    return n*(d_c+d_p), n*2.0*g, n*t

## 5. Delta-hedged Gamma-scalping backtest

In [ ]:
# BACKTEST LOOP (DAILY SAFE)
# =========================
records = []
q_fut = 0.0
prev_delta, prev_gamma, prev_theta = None, None, None
prev_S, curr_day, daily_pnl = None, None, 0.0

for row in es.itertuples(index=True):
    ts, S, sigma, K, T, day = row.Index, row.ES, row.sigma, row.K, row.T, row.day
    if prev_S is None:
        curr_day = day
        d, g, t = straddle_greeks(S, K, sigma, T, N_STRADDLES)
        prev_delta, prev_gamma, prev_theta = d, g, t
        prev_S = S
        continue

    dS = S - prev_S
    delta_step = prev_delta * dS
    gamma_step = 0.5 * prev_gamma * (dS**2)
    theta_step = prev_theta * dt_year
    hedge_step = q_fut * ES_POINT_VALUE * dS
    pnl_step   = delta_step + gamma_step + theta_step + hedge_step
    daily_pnl += pnl_step

    event = "HOLD"; commission = 0.0
    if (daily_pnl >= DAILY_PROFIT_TARGET) or (daily_pnl <= DAILY_DRAWDOWN_LIMIT):
        q_fut = 0.0; event = "STOP"
    else:
        d_new, g_new, t_new = straddle_greeks(S, K, sigma, T, N_STRADDLES)
        net_delta = d_new + q_fut * ES_POINT_VALUE
        denom = max(1e-6, abs(d_new))
        if (abs(net_delta)/denom) > DELTA_REHEDGE_BAND:
            q_target   = -d_new / ES_POINT_VALUE
            dq         = q_target - q_fut
            commission = COMMISSION_PER_ES * abs(dq)
            q_fut      = q_target
            event      = "REHEDGE"
        prev_delta, prev_gamma, prev_theta = d_new, g_new, t_new

    records.append([ts, S, delta_step, gamma_step, theta_step,
                    hedge_step, -commission, pnl_step, daily_pnl, q_fut, event])

    prev_S = S


## 6. Results

In [ ]:
# RESULTS
# =========================
res_cols = ["ts","S","delta_step","gamma_step","theta_step",
            "hedge_pnl","commission","pnl_step","daily_pnl","q_fut","event"]
res = pd.DataFrame(records, columns=res_cols)
res["ts"] = pd.to_datetime(res["ts"])
res = res.set_index("ts").sort_index()
res["gamma_scalp"] = res["delta_step"] + res["gamma_step"]

daily = res.groupby(pd.Grouper(freq="1D")).agg(
    gamma_scalp = ("gamma_scalp","sum"),
    theta       = ("theta_step","sum"),
    hedge_pnl   = ("hedge_pnl","sum"),
    commissions = ("commission","sum"),
    pnl_total   = ("daily_pnl","last"),
    rehedges    = ("event", lambda s: (s == "REHEDGE").sum())
)
daily["cum_pnl"] = daily["pnl_total"].cumsum()

print("PnL units: USD")
print(daily)
print("Total PnL USD:", round(daily["pnl_total"].sum(), 2))
print("Total Rehedges:", int(daily["rehedges"].sum()))

## 7. Visualisation

In [ ]:
# PLOTS
# =========================
plt.figure()
plt.bar(daily.index, daily["pnl_total"])
plt.title("Daily PnL (USD)")
plt.xlabel("Date"); plt.ylabel("PnL (USD)")
plt.tight_layout(); plt.show()

plt.figure()
plt.plot(daily.index, daily["cum_pnl"], marker="o")
plt.title("Cumulative PnL (USD)")
plt.xlabel("Date"); plt.ylabel("Cumulative PnL (USD)")
plt.tight_layout(); plt.show()

ax = daily[["gamma_scalp","theta","hedge_pnl","commissions"]].plot(kind="bar", figsize=(12,5), stacked=True)
ax.set_title("Daily PnL Components (USD)")
ax.set_xlabel("Date"); ax.set_ylabel("USD")
plt.tight_layout(); plt.show()


## 8. Limitations, Extensions